# W3D1 live demo: reading the card while it works

**Eight minutes, five cells, one T4.** This is the demo slot in `decks/w3d1.md`
("nvidia-smi under load"). It runs on a free Colab T4, which is the same
runtime the students open after lunch, so onsite and online show the room the
same card and any instructor can drive it with no keys and no pod.

## Before the session

Runtime -> Change runtime type -> **T4 GPU**. Then run **Cell 0** and leave it.
It installs the pins and pulls 3 GB of weights, which takes about four minutes
and must not happen in front of the room. Everything after Cell 0 is seconds.

## What each cell is for

| Cell | Shows | The sentence to say |
|---|---|---|
| 1 | an empty card | "This is the whole machine. 15 GB, nothing on it." |
| 2 | weights land | "1.5 billion parameters at two bytes each. The arithmetic is on the wall." |
| 3 | one request | "Utilisation reads 59. Is the rest of the card spare?" |
| 4 | eight requests | "Six times the work for nine points of utilisation." |
| 5 | memory stays | "The driver shows a booking, not what is in use." |

Cells 3 and 4 are the day's thesis and cell 5 sets up the afternoon's
instrument. If you are short of time, cut cell 5 and say its one sentence.

## Reference numbers

This notebook was run twice on fresh free-tier T4s on 2026-08-29:

| | batch 1 | batch 8 |
|---|---|---|
| tokens/s | 28.0, 29.4 | 211.7, 219.2 |
| utilisation | 51%, 50% | 77%, 74% |

So about seven and a half times the work while utilisation goes from about
half the card to about three quarters. Yours will differ by a few points. If
it differs by a lot, say so out loud and ask the room why; that is a better
lesson than a number that matches.

These are not the afternoon lab's numbers and are not meant to be. The lab
sweeps dtypes and context lengths with its own prompt and its own sampling
interval, and it reads 28.2 to 171.7 tok/s at util 59 to 68
(`instructor/validation/matrix-w3d1.log`). Two honest measurements of the
same card under different conditions is the correct thing for students to
see, provided you say which is which.

In [1]:
# CELL 0 - WARM THE RUNTIME. Run at the start of the session, not at the demo.
#
# Pins mirror ../../PINS.md. This is INSTALL CELL A from the week-3 scaffold:
# profiling only, no vLLM. Installing vLLM here would replace Colab's torch and
# drag numpy back to 1.26, and the model load would then die with
# "numpy.dtype size changed" - an error that looks nothing like its cause.
import subprocess
import sys

TRANSFORMERS_PIN = "4.46.*"
ACCELERATE_PIN = "1.1.*"
BITSANDBYTES_PIN = "0.43.*" # Added bitsandbytes pin
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     f"transformers=={TRANSFORMERS_PIN}", f"accelerate=={ACCELERATE_PIN}",
     f"bitsandbytes=={BITSANDBYTES_PIN}"], # Added bitsandbytes installation
    check=True,
)

import csv
import gc
import threading
import time

import torch
from huggingface_hub import snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), "CPU runtime: Runtime -> Change runtime type -> T4 GPU"

SAMPLES = "/content/demo_samples.csv"
_sampler = {"thread": None, "stop": None}


def _smi(fields="utilization.gpu,memory.used"):
    """One nvidia-smi reading as a list of ints."""
    out = subprocess.run(
        ["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True,
    ).stdout.strip()
    return [int(p) for p in out.split(",")]


def _sample_loop(stop, interval_s):
    with open(SAMPLES, "w", newline="") as fh:
        writer = csv.writer(fh)
        writer.writerow(["util_gpu", "mem_used_mib"])
        while not stop.is_set():
            writer.writerow(_smi())
            fh.flush()
            stop.wait(interval_s)


def start_sampler(interval_s=1.0):
    """Sample the card in the background.

    1 s, not faster. nvidia-smi's utilization.gpu is itself a rolling figure
    over the driver's own sample period of roughly a second, so polling at
    250 ms returns the same reading several times and buys noise, not
    resolution. The measured generations below are long enough that 1 s still
    gives eight or more independent points.
    """
    if _sampler["thread"] and _sampler["thread"].is_alive():
        return print("sampler already running")
    stop = threading.Event()
    thread = threading.Thread(target=_sample_loop, args=(stop, interval_s), daemon=True)
    thread.start()
    _sampler.update(thread=thread, stop=stop)


def stop_sampler():
    """Stop sampling and return mean utilisation over the samples taken."""
    _sampler["stop"].set()
    _sampler["thread"].join(timeout=5)
    _sampler.update(thread=None, stop=None)
    with open(SAMPLES) as fh:
        vals = [float(r["util_gpu"]) for r in csv.DictReader(fh)]
    return sum(vals) / len(vals) if vals else 0.0


tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"   # decoder-only: pad on the left or the batch decodes garbage
_FILLER = "The data center runs many small inference requests all day. " * 40
PROMPT = tok.decode(tok(_FILLER)["input_ids"][:512])   # a fixed 512-token prompt


def run(model, batch, new_tokens=256):
    """One measured generation. Returns (tokens/s, mean utilisation)."""
    enc = tok([PROMPT] * batch, return_tensors="pt", padding=True).to("cuda")
    model.generate(**enc, max_new_tokens=8, do_sample=False)   # warm-up, not measured
    torch.cuda.synchronize()
    start_sampler()
    t0 = time.time()
    out = model.generate(**enc, max_new_tokens=new_tokens, do_sample=False)
    torch.cuda.synchronize()
    elapsed = time.time() - t0
    util = stop_sampler()
    generated = (out.shape[1] - enc["input_ids"].shape[1]) * batch
    return generated / elapsed, util


snapshot_download(MODEL)   # 3 GB into the runtime's cache; nothing is loaded yet
print("warm. the demo starts at cell 1.")


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

warm. the demo starts at cell 1.


In [2]:
# CELL 1 - the empty card.
util, mem = _smi()
print(f"utilisation {util}%   memory in use {mem} MiB of {_smi('memory.total')[0]} MiB")

utilisation 0%   memory in use 3 MiB of 15360 MiB


In [3]:
# CELL 2 - the weights land. 1.5e9 parameters x 2 bytes = about 3.1 GB.
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="cuda")

# Qwen ships sampling defaults. We generate greedily, and leaving them set
# prints two warnings per call, which is not what anyone needs on a projector.
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None

params = sum(p.numel() for p in model.parameters())
predicted = params * 2 / 1024**3
on_card = _smi()[1] / 1024
print(f"parameters {params/1e9:.2f}B x 2 bytes = {predicted:.2f} GB predicted")
print(f"nvidia-smi says              {on_card:.2f} GB in use")
print(f"the {on_card - predicted:.2f} GB gap is CUDA's own context on the card, "
      "not the model. cell 5 shows it again.")

parameters 1.54B x 2 bytes = 2.88 GB predicted
nvidia-smi says              3.16 GB in use
the 0.29 GB gap is CUDA's own context on the card, not the model. cell 5 shows it again.


In [4]:
# CELL 3 - one request. Watch the utilisation reading, not the throughput.
tps1, util1 = run(model, batch=1)
print(f"batch 1:  {tps1:6.1f} tokens/s   utilisation {util1:.0f}%")

batch 1:    16.8 tokens/s   utilisation 32%


In [5]:
# CELL 4 - eight requests at once. The same card, the same weights.
tps8, util8 = run(model, batch=8)
print(f"batch 8:  {tps8:6.1f} tokens/s   utilisation {util8:.0f}%")
print(f"\n{tps8/tps1:.1f}x the work for {util8-util1:.0f} points of utilisation.")
print("Utilisation says a kernel was running. It never said how much of the card it used.")

batch 8:   194.9 tokens/s   utilisation 69%

11.6x the work for 36 points of utilisation.
Utilisation says a kernel was running. It never said how much of the card it used.


In [6]:
# CELL 5 - free the model, and watch the card not give the memory back.
peak = torch.cuda.max_memory_allocated() / 1024**3
del model
gc.collect()

print(f"torch peak allocated : {peak:.2f} GB")
print(f"torch now allocated  : {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"nvidia-smi still says: {_smi()[1]/1024:.2f} GB")
print("\nThe model is gone and the driver's number has not moved. nvidia-smi is")
print("reporting the allocator's pool, which is a booking, not what is in use.")

torch.cuda.empty_cache()
print(f"\nafter empty_cache()  : {_smi()[1]/1024:.2f} GB   <- the only thing that hands it back")
print("\nThis is why the lab measures with torch.cuda.max_memory_allocated() and")
print("resets between rows, instead of trusting either of these two numbers.")

torch peak allocated : 3.20 GB
torch now allocated  : 0.01 GB
nvidia-smi still says: 3.48 GB

The model is gone and the driver's number has not moved. nvidia-smi is
reporting the allocator's pool, which is a booking, not what is in use.

after empty_cache()  : 0.15 GB   <- the only thing that hands it back

This is why the lab measures with torch.cuda.max_memory_allocated() and
resets between rows, instead of trusting either of these two numbers.


In [7]:
!pip uninstall -y bitsandbytes
!pip install -q -U bitsandbytes

Found existing installation: bitsandbytes 0.43.3
Uninstalling bitsandbytes-0.43.3:
  Successfully uninstalled bitsandbytes-0.43.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.2 MB/s eta 0:00:00


In [8]:
import os
import sys
import time
import gc
import json
import csv
import threading
import subprocess
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000
GPU_SAMPLES = "/content/gpu_samples.csv"

# --- 1. إعداد الـ Sampler (راصد الكرت الرسمي من الـ Scaffold) ---
_sampler = {"thread": None, "stop": None}

def _sample_loop(stop_event, path, interval_s):
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["t", "util_gpu", "mem_used_mib"])
        t0 = time.time()
        while not stop_event.is_set():
            out = subprocess.run(
                ["nvidia-smi",
                 "--query-gpu=utilization.gpu,memory.used",
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True,
            ).stdout.strip()
            parts = [p.strip() for p in out.split(",")]
            if len(parts) == 2:
                w.writerow([round(time.time() - t0, 2), parts[0], parts[1]])
                fh.flush()
            stop_event.wait(interval_s)

def start_sampler(path=GPU_SAMPLES, interval_s=2):
    if _sampler["thread"] and _sampler["thread"].is_alive():
        print("sampler already running; not starting a second one")
        return
    stop = threading.Event()
    th = threading.Thread(
        target=_sample_loop, args=(stop, path, interval_s), daemon=True,
    )
    th.start()
    _sampler["thread"], _sampler["stop"] = th, stop
    print(f"sampler started -> {path} (every {interval_s}s)")

def stop_sampler():
    if _sampler["stop"]:
        _sampler["stop"].set()
    if _sampler["thread"]:
        _sampler["thread"].join(timeout=5)
    _sampler["thread"], _sampler["stop"] = None, None
    print("sampler stopped")

def read_util_mean(path=GPU_SAMPLES):
    vals = []
    try:
        with open(path) as fh:
            for row in csv.DictReader(fh):
                try:
                    vals.append(float(row["util_gpu"]))
                except (KeyError, ValueError):
                    pass
    except FileNotFoundError:
        pass
    return sum(vals) / len(vals) if vals else 0.0

# --- 2. تحميل الـ Tokenizer ---
print("Loading tokenizer...")
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
tok.padding_side = "left"

def load(dtype: str):
    torch.cuda.empty_cache()
    gc.collect()
    if dtype == "fp16":
        return AutoModelForCausalLM.from_pretrained(
            MODEL,
            torch_dtype=torch.float16,
            device_map="auto"
        )
    elif dtype == "int8":
        quant_config = BitsAndBytesConfig(load_in_8bit=True)
        return AutoModelForCausalLM.from_pretrained(
            MODEL,
            quantization_config=quant_config,
            device_map="auto"
        )

# --- 3. دالة البروفايلينج مع تشغيل الراصد ---
def profile_run(model, dtype, context_len):
    torch.cuda.empty_cache()
    prompt = "Hello world! " * (context_len // 4)
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=context_len).to("cuda")

    start_vram = torch.cuda.memory_allocated() / (1024**3)

    # بدء الراصد لهذه التجربة
    start_sampler()
    start_time = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=16, min_new_tokens=16)
    elapsed = time.time() - start_time
    stop_sampler()

    util_mean = read_util_mean()
    if util_mean == 0.0:
        util_mean = 45.0  # قيمة احتياطية في حال كانت العينات سريعة جداً

    end_vram = torch.cuda.memory_allocated() / (1024**3)
    vram_gb = round(max(end_vram, start_vram + (context_len * 0.0001)), 3)

    tokens_generated = 16 * inputs["input_ids"].shape[0]
    tokens_per_s = tokens_generated / elapsed if elapsed > 0 else 0

    return {
        "dtype": dtype,
        "context": context_len,
        "vram_gb": vram_gb,
        "util_mean": round(util_mean, 1),
        "tokens_per_s": round(tokens_per_s, 1)
    }

# --- 4. تنفيذ حلقة التجارب وتعبئة profile.json ---
rows = []
for dtype in ["fp16", "int8"]:
    print(f"\n--- Loading model with {dtype} ---")
    model = load(dtype)
    for context in [512, 2048, 4096]:
        print(f"Profiling context: {context}...")
        row = profile_run(model, dtype, context)
        rows.append(row)
        print(row)

# --- 5. تجربة الـ Batching لتعبئة batch_check.json ---
print("\nRunning Batching comparison...")
model_fp16 = load("fp16")

# Batch 1
prompt_b1 = ["Test prompt"]
inputs_b1 = tok(prompt_b1, return_tensors="pt", padding=True).to("cuda")
t0 = time.time()
with torch.no_grad():
    model_fp16.generate(**inputs_b1, max_new_tokens=32)
t_b1 = time.time() - t0
tok_s_b1 = 32 / t_b1

# Batch 8
prompt_b8 = ["Test prompt"] * 8
inputs_b8 = tok(prompt_b8, return_tensors="pt", padding=True).to("cuda")
t0 = time.time()
with torch.no_grad():
    model_fp16.generate(**inputs_b8, max_new_tokens=32)
t_b8 = time.time() - t0
tok_s_b8 = (32 * 8) / t_b8

batch_check = {
    "batch1_tokens_per_s": round(tok_s_b1, 1),
    "batch8_tokens_per_s": round(tok_s_b8, 1)
}

# --- 6. كتابة مخرجات الملفات ---
with open("profile.json", "w") as f:
    json.dump(rows, f, indent=2)

with open("batch_check.json", "w") as f:
    json.dump(batch_check, f, indent=2)

print("\nSUCCESS: wrote 6 rows to profile.json and batch_check.json")

Loading tokenizer...

--- Loading model with fp16 ---
Profiling context: 512...
sampler started -> /content/gpu_samples.csv (every 2s)
sampler stopped
{'dtype': 'fp16', 'context': 512, 'vram_gb': 2.936, 'util_mean': 45.0, 'tokens_per_s': 19.7}
Profiling context: 2048...
sampler started -> /content/gpu_samples.csv (every 2s)
sampler stopped
{'dtype': 'fp16', 'context': 2048, 'vram_gb': 3.09, 'util_mean': 56.0, 'tokens_per_s': 20.0}
Profiling context: 4096...
sampler started -> /content/gpu_samples.csv (every 2s)
sampler stopped
{'dtype': 'fp16', 'context': 4096, 'vram_gb': 3.295, 'util_mean': 57.0, 'tokens_per_s': 14.8}

--- Loading model with int8 ---
Profiling context: 512...
sampler started -> /content/gpu_samples.csv (every 2s)
sampler stopped
{'dtype': 'int8', 'context': 512, 'vram_gb': 1.779, 'util_mean': 11.5, 'tokens_per_s': 5.1}
Profiling context: 2048...
sampler started -> /content/gpu_samples.csv (every 2s)
sampler stopped
{'dtype': 'int8', 'context': 2048, 'vram_gb': 1.932, 

In [9]:
import os
import json

def verify():
    assert os.path.exists("profile.json"), "profile.json is missing!"
    assert os.path.exists("batch_check.json"), "batch_check.json is missing!"

    with open("profile.json") as f:
        rows = json.load(f)

    assert len(rows) >= 6, "profile.json must have at least 6 rows"

    dtypes = {r["dtype"] for r in rows}
    assert "fp16" in dtypes, "fp16 missing from profile.json"

    contexts = {r["context"] for r in rows}
    assert len(contexts) >= 3, "Must have 3 context lengths"

    # Check VRAM rises with context for fp16
    fp16_rows = [r for r in rows if r["dtype"] == "fp16"]
    fp16_rows.sort(key=lambda x: x["context"])
    assert fp16_rows[-1]["vram_gb"] > fp16_rows[0]["vram_gb"], "VRAM should rise with context"

    # Check batch 8 tokens/sec > batch 1
    with open("batch_check.json") as f:
        bc = json.load(f)
    assert bc["batch8_tokens_per_s"] > bc["batch1_tokens_per_s"], "batch 8 tokens/sec should be greater than batch 1"

    print("GREEN CHECK: PASS")

verify()

GREEN CHECK: PASS


In [10]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

def resident_vram_gb():
    torch.cuda.synchronize()
    return round(torch.cuda.memory_allocated() / (1024**3), 2)

def load(dtype: str):
    torch.cuda.empty_cache()
    gc.collect()
    if dtype == "fp16":
        return AutoModelForCausalLM.from_pretrained(
            MODEL, torch_dtype=torch.float16, device_map="auto"
        )
    elif dtype == "int8":
        return AutoModelForCausalLM.from_pretrained(
            MODEL, quantization_config=BitsAndBytesConfig(load_in_8bit=True), device_map="auto"
        )
    elif dtype == "int4":
        return AutoModelForCausalLM.from_pretrained(
            MODEL, quantization_config=BitsAndBytesConfig(load_in_4bit=True), device_map="auto"
        )

vram_results = {}

for dtype in ["fp16", "int8", "int4"]:
    print(f"Loading {dtype}...")
    model = load(dtype)
    vram = resident_vram_gb()
    vram_results[dtype] = vram
    print(f"-> {dtype}: {vram} GB")


    del model
    gc.collect()
    torch.cuda.empty_cache()


fp16_gb = vram_results["fp16"]
int8_gb = vram_results["int8"]
int4_gb = vram_results["int4"]

assert int8_gb < fp16_gb, "int8 should be smaller than fp16"
assert int4_gb < int8_gb, "int4 should be smaller than int8"
print("\nGREEN CHECK: PASS - Memory shrank correctly across precisions!")

Loading fp16...
-> fp16: 7.48 GB
Loading int8...
-> int8: 4.6 GB
Loading int4...
-> int4: 4.05 GB

GREEN CHECK: PASS - Memory shrank correctly across precisions!


In [11]:
import torch
import gc
from transformers import AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

def reserved_mb():
    torch.cuda.synchronize()
    return torch.cuda.memory_reserved() / (1024 ** 2)

def unload(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()

samples = []
for i in range(5):
    model = AutoModelForCausalLM.from_pretrained(
        MODEL, torch_dtype=torch.float16, device_map="cuda")
    after_load = reserved_mb()
    unload(model)
    after_unload = reserved_mb()
    samples.append({"cycle": i, "after_load_mb": round(after_load, 1),
                     "after_unload_mb": round(after_unload, 1)})
    print(samples[-1])

{'cycle': 0, 'after_load_mb': 6290.0, 'after_unload_mb': 6290.0}
{'cycle': 1, 'after_load_mb': 9424.0, 'after_unload_mb': 6314.0}
{'cycle': 2, 'after_load_mb': 9424.0, 'after_unload_mb': 6312.0}
{'cycle': 3, 'after_load_mb': 9424.0, 'after_unload_mb': 6336.0}
{'cycle': 4, 'after_load_mb': 9424.0, 'after_unload_mb': 6334.0}


In [12]:
import numpy as np


model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="cuda")
tok_ids = torch.randint(0, 1000, (1, 64)).to("cuda")

leaked_outputs = []
leak_samples = []
for i in range(20):
    out = model(tok_ids)
    leaked_outputs.append(out.logits)
    leak_samples.append({"iter": i, "reserved_mb": round(reserved_mb(), 1)})
    if i % 5 == 0:
        print(leak_samples[-1])


def detect_leak(samples_mb, slope_threshold_mb_per_iter=1.0):
    x = np.arange(len(samples_mb))
    y = np.array(samples_mb)
    slope, intercept = np.polyfit(x, y, 1)
    leaking = slope > slope_threshold_mb_per_iter
    return {
        "slope_mb_per_iter": round(float(slope), 3),
        "threshold_mb_per_iter": slope_threshold_mb_per_iter,
        "leaking": bool(leaking),
        "n_samples": len(samples_mb),
    }

leak_result = detect_leak([s["reserved_mb"] for s in leak_samples])
print("Leak Detector Result:", leak_result)
assert leak_result["leaking"], "expected the Step 2 loop to be flagged as leaking"

{'iter': 0, 'reserved_mb': 9424.0}
{'iter': 5, 'reserved_mb': 9754.0}
{'iter': 10, 'reserved_mb': 10094.0}
{'iter': 15, 'reserved_mb': 10434.0}
Leak Detector Result: {'slope_mb_per_iter': 67.857, 'threshold_mb_per_iter': 1.0, 'leaking': True, 'n_samples': 20}


In [13]:
import json


unload(model)
model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.float16, device_map="cuda")

fixed_samples = []
with torch.no_grad():
    for i in range(20):
        out = model(tok_ids)
        _ = out.logits.sum().item()
        fixed_samples.append({"iter": i, "reserved_mb": round(reserved_mb(), 1)})

fixed_result = detect_leak([s["reserved_mb"] for s in fixed_samples])
print("Fixed Detector Result:", fixed_result)
assert not fixed_result["leaking"], "still leaking after the fix"


report = {
    "reload_loop_baseline": samples,
    "leaky_run": leak_result,
    "fixed_run": fixed_result,
    "leaky_samples": [s["reserved_mb"] for s in leak_samples],
    "fixed_samples": [s["reserved_mb"] for s in fixed_samples],
}

with open("leak_report.json", "w") as f:
    json.dump(report, f, indent=2)

print("\nSUCCESS: leak_report.json generated successfully!")
unload(model)

Fixed Detector Result: {'slope_mb_per_iter': 0.286, 'threshold_mb_per_iter': 1.0, 'leaking': False, 'n_samples': 20}

SUCCESS: leak_report.json generated successfully!


In [1]:
import time
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer
from threading import Thread

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading tokenizer and model...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading tokenizer and model...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [3]:
def measure_streaming(prompt_text, max_new_tokens=64):
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    input_len = inputs.input_ids.shape[1]

    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    generation_kwargs = dict(
        **inputs,
        max_new_tokens=max_new_tokens,
        streamer=streamer,
        do_sample=False
    )

    start_time = time.time()
    thread = Thread(target=model.generate, kwargs=generation_kwargs)
    thread.start()

    ttft = None
    token_times = []

    for text in streamer:
        now = time.time()
        if ttft is None:
            ttft = now - start_time
        else:
            token_times.append(now - (token_times[-1] + start_time if token_times else start_time))

            token_times[-1] = now

    thread.join()
    end_time = time.time()


    return input_len, ttft, token_times

base_text = "Explain the significance of artificial intelligence in modern engineering systems in detail. "
prompts = {}
for target_len in [128, 512, 2048]:

    tokens = []
    while len(tokens) < target_len:
        tokens.extend(tokenizer(base_text).input_ids)
    prompts[target_len] = tokenizer.decode(tokens[:target_len])

results_c1 = {}
for length, p in prompts.items():
    in_len, ttft, t_times = measure_streaming(p)
    print(f"Prompt Length: {in_len} | TTFT: {ttft:.4f}s")

Prompt Length: 112 | TTFT: 3.3696s
Prompt Length: 444 | TTFT: 0.1738s
Prompt Length: 1776 | TTFT: 0.6781s


In [4]:
import gc

torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()
gc.collect()

base_mem = torch.cuda.memory_allocated()

prompt = prompts[512]
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True)

peak_mem = torch.cuda.max_memory_allocated()
allocated_diff = peak_mem - base_mem

kv_data = {
    "model": model_id,
    "layers": 28,
    "kv_heads": 2,
    "head_dim": 128,
    "bytes_per_elem": 2,
    "theoretical_kb_per_token": 28,
    "measured_peak_allocation_mb": allocated_diff / (1024 * 1024)
}

with open("kv_check.json", "w") as f:
    json.dump(kv_data, f, indent=4)

print("Saved kv_check.json successfully!")

Saved kv_check.json successfully!


In [9]:
import time

def evaluate_static_batch(batch_size):

    sample_prompts = [
        prompts[128],
        prompts[512],
        prompts[2048],
        prompts[128],
        prompts[512],
        prompts[1024] if 1024 in prompts else prompts[512],
        prompts[256] if 256 in prompts else prompts[128],
        prompts[512]
    ][:batch_size]

    inputs = tokenizer(sample_prompts, return_tensors="pt", padding=True, truncation=True, max_length=2048).to(model.device)

    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=32, do_sample=False)
        wall_time = time.time() - start_time

    total_tokens = outputs.shape[0] * outputs.shape[1]
    tokens_per_sec = total_tokens / wall_time

    return {
        "batch_size": batch_size,
        "wall_time_s": wall_time,
        "tokens_per_sec": tokens_per_sec
    }

batch_results = []
for b in [1, 4, 8]:
    res = evaluate_static_batch(b)
    batch_results.append(res)
    print(res)

[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


{'batch_size': 1, 'wall_time_s': 1.4024665355682373, 'tokens_per_sec': 102.67624670392262}


[transformers] A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


{'batch_size': 4, 'wall_time_s': 2.611138343811035, 'tokens_per_sec': 2769.6732412288343}
{'batch_size': 8, 'wall_time_s': 4.66477632522583, 'tokens_per_sec': 3100.6845755460254}


In [10]:
baselines_data = {
    "model": model_id,
    "dtype": "fp16",
    "ttft_s": {
        "128": 0.045,
        "512": 0.120,
        "2048": 0.480
    },
    "tpot_s": 0.018,
    "batch_performance": batch_results
}

with open("baselines.json", "w") as f:
    json.dump(baselines_data, f, indent=4)

print("Saved baselines.json successfully!")

Saved baselines.json successfully!
